# NFL Moneyline Research Notebook — Clean Rebuild

This notebook rebuilds the project from scratch with **V2.0 restored as the active champion**.

### Version registry
- **V1.0** — baseline full-game score simulator
- **V2.0** — leakage-safe probability calibration ✅ **ACTIVE CHAMPION**
- **V2.1** — matchup-specific pace ❌ rejected
- **V2.2** — starting field position ❌ rejected
- **V2.3** — turnover type + defensive TDs ❌ rejected
- **V2.4** — next experiment, not yet implemented

The rejected experiments are preserved later in the notebook for research history, but **they are not carried into the active model**.


In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

!pip install -q nflreadpy polars pandas pyarrow numpy scipy scikit-learn matplotlib

import math
import time

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import nflreadpy as nfl

from sklearn.metrics import brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression

MODEL_NAME = "NFL Moneyline"
ACTIVE_MODEL_VERSION = "V2.0"

START_SEASON = 2018
CURRENT_SEASON = 2026
PRIOR_GAMES_EQUIVALENT = 6

TEAM_MAP = {
    "OAK": "LV",
    "STL": "LA",
    "SD": "LAC",
    "JAC": "JAX",
    "WSH": "WAS",
}

print(f"{MODEL_NAME} — active model {ACTIVE_MODEL_VERSION}")
print("Environment ready.")


In [ ]:
# ============================================================
# CELL 2 — LOAD + NORMALIZE NFL PLAY-BY-PLAY
# ============================================================

SEASONS = list(range(START_SEASON, CURRENT_SEASON + 1))
print("Loading seasons:", SEASONS)

pbp = nfl.load_pbp(SEASONS)

if isinstance(pbp, pl.LazyFrame):
    pbp = pbp.collect()

print("Raw PBP shape:", pbp.shape)

for col in ["posteam", "defteam", "home_team", "away_team"]:
    if col in pbp.columns:
        pbp = pbp.with_columns(
            pl.col(col).replace(TEAM_MAP).alias(col)
        )

pbp_reg = pbp.filter(
    pl.col("season_type") == "REG"
)

season_check = (
    pbp_reg
    .group_by("season")
    .agg(pl.col("game_id").n_unique().alias("games"))
    .sort("season")
)

display(season_check)


In [ ]:
# ============================================================
# CELL 3 — MASTER GAME TABLE
# One row per regular-season game
# ============================================================

games = (
    pbp_reg
    .filter(
        pl.col("total_home_score").is_not_null() &
        pl.col("total_away_score").is_not_null()
    )
    .group_by("game_id")
    .agg([
        pl.col("season").first().alias("season"),
        pl.col("week").first().alias("week"),
        pl.col("home_team").first().alias("home_team"),
        pl.col("away_team").first().alias("away_team"),
        pl.col("total_home_score").max().alias("home_points"),
        pl.col("total_away_score").max().alias("away_points"),
    ])
    .with_columns([
        (pl.col("home_points") + pl.col("away_points")).alias("total_points"),
        (pl.col("home_points") - pl.col("away_points")).alias("home_margin"),
        (pl.col("home_points") > pl.col("away_points")).cast(pl.Int8).alias("home_win"),
        (pl.col("away_points") > pl.col("home_points")).cast(pl.Int8).alias("away_win"),
        (pl.col("home_points") == pl.col("away_points")).cast(pl.Int8).alias("tie"),
    ])
    .sort(["season", "week", "game_id"])
)

print("Games:", games.height)
display(games.tail(10))


In [ ]:
# ============================================================
# CELL 4 — MASTER DRIVE TABLE
# One row per actual offensive possession
# ============================================================

drive_source = (
    pbp_reg
    .filter(
        pl.col("drive").is_not_null() &
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null()
    )
    .with_columns([
        (
            (pl.col("interception").fill_null(0) == 1) |
            (pl.col("fumble_lost").fill_null(0) == 1)
        )
        .cast(pl.Int8)
        .alias("_turnover_play")
    ])
)

drive_aggs = [
    pl.col("season").first().alias("season"),
    pl.col("week").first().alias("week"),
    pl.col("home_team").first().alias("home_team"),
    pl.col("away_team").first().alias("away_team"),
    pl.col("posteam").first().alias("offense"),
    pl.col("defteam").first().alias("defense"),
    pl.col("_turnover_play").max().alias("turnover"),
    pl.col("posteam_score").drop_nulls().first().alias("start_offense_score"),
    pl.col("posteam_score_post").drop_nulls().last().alias("end_offense_score"),
]

if "game_seconds_remaining" in pbp_reg.columns:
    drive_aggs.extend([
        pl.col("game_seconds_remaining").drop_nulls().first().alias("drive_start_seconds"),
        pl.col("game_seconds_remaining").drop_nulls().last().alias("drive_end_seconds"),
    ])

drives = (
    drive_source
    .sort(["game_id", "drive", "play_id"])
    .group_by(["game_id", "drive"])
    .agg(drive_aggs)
    .with_columns([
        (
            pl.col("end_offense_score") -
            pl.col("start_offense_score")
        )
        .fill_null(0)
        .clip(lower_bound=0)
        .alias("drive_points")
    ])
    .with_columns([
        pl.when(pl.col("drive_points").is_between(6, 8))
          .then(pl.lit("TD"))
          .when(pl.col("drive_points") == 3)
          .then(pl.lit("FG"))
          .when(pl.col("turnover") == 1)
          .then(pl.lit("TURNOVER"))
          .otherwise(pl.lit("EMPTY"))
          .alias("v1_outcome")
    ])
    .sort(["season", "week", "game_id", "drive"])
)

print("Drive rows:", drives.height)


In [ ]:
# ============================================================
# CELL 5 — DRIVE SANITY CHECK
# ============================================================

historical_drives = drives.filter(
    pl.col("season") < CURRENT_SEASON
)

outcome_summary = (
    historical_drives
    .group_by("v1_outcome")
    .agg([
        pl.len().alias("drives"),
        pl.col("drive_points").mean().alias("avg_raw_points"),
    ])
    .with_columns(
        (pl.col("drives") / pl.col("drives").sum()).alias("rate")
    )
    .sort("rate", descending=True)
)

display(outcome_summary)

print("\nTD scoring distribution:")
display(
    historical_drives
    .filter(pl.col("v1_outcome") == "TD")
    .group_by("drive_points")
    .agg(pl.len().alias("drives"))
    .sort("drive_points")
)

print("\nDrives with > 8 raw offensive points:")
print(
    historical_drives
    .filter(pl.col("drive_points") > 8)
    .height
)

drive_count_summary = (
    historical_drives
    .group_by("game_id")
    .agg(pl.len().alias("total_drives"))
)

print("\nAverage drives per completed game:")
print(round(drive_count_summary["total_drives"].mean(), 3))


In [ ]:
# ============================================================
# CELL 6 — CLEAN V1 MODELING DRIVES
# TD=7, FG=3, everything else=0
# ============================================================

v1_drives = (
    drives
    .filter(
        pl.col("drive_points").is_in([0, 3, 6, 7, 8])
    )
    .with_columns([
        pl.when(pl.col("v1_outcome") == "TD")
          .then(pl.lit(7))
          .when(pl.col("v1_outcome") == "FG")
          .then(pl.lit(3))
          .otherwise(pl.lit(0))
          .alias("v1_sim_points")
    ])
)

print("Master drives:", drives.height)
print("V1 modeling drives:", v1_drives.height)
print("Excluded special/strange drives:", drives.height - v1_drives.height)

display(
    v1_drives
    .group_by("v1_outcome")
    .agg([
        pl.len().alias("drives"),
        pl.col("v1_sim_points").mean().alias("sim_points"),
    ])
    .sort("drives", descending=True)
)


In [ ]:
# ============================================================
# CELL 7 — TEAM DRIVE-OUTCOME RATINGS
# ============================================================

def build_v1_team_ratings(drive_data):

    labeled = (
        drive_data
        .with_columns([
            (pl.col("v1_outcome") == "TD").cast(pl.Float64).alias("_td"),
            (pl.col("v1_outcome") == "FG").cast(pl.Float64).alias("_fg"),
            (pl.col("v1_outcome") == "TURNOVER").cast(pl.Float64).alias("_turnover"),
            (pl.col("v1_outcome") == "EMPTY").cast(pl.Float64).alias("_empty"),
        ])
    )

    offense = (
        labeled
        .group_by("offense")
        .agg([
            pl.col("game_id").n_unique().alias("games"),
            pl.len().alias("drives"),
            pl.col("_td").mean().alias("td_rate"),
            pl.col("_fg").mean().alias("fg_rate"),
            pl.col("_turnover").mean().alias("turnover_rate"),
            pl.col("_empty").mean().alias("empty_rate"),
        ])
        .rename({"offense": "team"})
        .sort("team")
    )

    defense = (
        labeled
        .group_by("defense")
        .agg([
            pl.col("game_id").n_unique().alias("games"),
            pl.len().alias("drives_faced"),
            pl.col("_td").mean().alias("td_rate_allowed"),
            pl.col("_fg").mean().alias("fg_rate_allowed"),
            pl.col("_turnover").mean().alias("turnover_rate_forced"),
            pl.col("_empty").mean().alias("empty_rate_allowed"),
        ])
        .rename({"defense": "team"})
        .sort("team")
    )

    return offense, defense

print("build_v1_team_ratings() ready.")


In [ ]:
# ============================================================
# CELL 8 — LEAKAGE-SAFE PREGAME TEAM RATINGS
# ============================================================

def build_v1_pregame_ratings(
    target_season,
    target_week,
    prior_games_equivalent=PRIOR_GAMES_EQUIVALENT
):
    prior_season = target_season - 1

    prior_data = v1_drives.filter(
        pl.col("season") == prior_season
    )

    current_data = v1_drives.filter(
        (pl.col("season") == target_season) &
        (pl.col("week") < target_week)
    )

    prior_off, prior_def = build_v1_team_ratings(prior_data)
    current_off, current_def = build_v1_team_ratings(current_data)

    offense = (
        prior_off
        .rename({
            "games": "prior_games",
            "drives": "prior_drives",
            "td_rate": "prior_td_rate",
            "fg_rate": "prior_fg_rate",
            "turnover_rate": "prior_turnover_rate",
            "empty_rate": "prior_empty_rate",
        })
        .join(
            current_off.rename({
                "games": "current_games",
                "drives": "current_drives",
                "td_rate": "current_td_rate",
                "fg_rate": "current_fg_rate",
                "turnover_rate": "current_turnover_rate",
                "empty_rate": "current_empty_rate",
            }),
            on="team",
            how="left"
        )
        .with_columns([
            pl.col("current_games").fill_null(0),
            pl.col("current_drives").fill_null(0),
            pl.col("current_td_rate").fill_null(pl.col("prior_td_rate")),
            pl.col("current_fg_rate").fill_null(pl.col("prior_fg_rate")),
            pl.col("current_turnover_rate").fill_null(pl.col("prior_turnover_rate")),
            pl.col("current_empty_rate").fill_null(pl.col("prior_empty_rate")),
        ])
        .with_columns([
            (
                (
                    pl.col("prior_td_rate") * prior_games_equivalent +
                    pl.col("current_td_rate") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("td_rate"),

            (
                (
                    pl.col("prior_fg_rate") * prior_games_equivalent +
                    pl.col("current_fg_rate") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("fg_rate"),

            (
                (
                    pl.col("prior_turnover_rate") * prior_games_equivalent +
                    pl.col("current_turnover_rate") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("turnover_rate"),

            (
                (
                    pl.col("prior_empty_rate") * prior_games_equivalent +
                    pl.col("current_empty_rate") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("empty_rate"),
        ])
    )

    defense = (
        prior_def
        .rename({
            "games": "prior_games",
            "drives_faced": "prior_drives_faced",
            "td_rate_allowed": "prior_td_rate_allowed",
            "fg_rate_allowed": "prior_fg_rate_allowed",
            "turnover_rate_forced": "prior_turnover_rate_forced",
            "empty_rate_allowed": "prior_empty_rate_allowed",
        })
        .join(
            current_def.rename({
                "games": "current_games",
                "drives_faced": "current_drives_faced",
                "td_rate_allowed": "current_td_rate_allowed",
                "fg_rate_allowed": "current_fg_rate_allowed",
                "turnover_rate_forced": "current_turnover_rate_forced",
                "empty_rate_allowed": "current_empty_rate_allowed",
            }),
            on="team",
            how="left"
        )
        .with_columns([
            pl.col("current_games").fill_null(0),
            pl.col("current_drives_faced").fill_null(0),
            pl.col("current_td_rate_allowed").fill_null(pl.col("prior_td_rate_allowed")),
            pl.col("current_fg_rate_allowed").fill_null(pl.col("prior_fg_rate_allowed")),
            pl.col("current_turnover_rate_forced").fill_null(pl.col("prior_turnover_rate_forced")),
            pl.col("current_empty_rate_allowed").fill_null(pl.col("prior_empty_rate_allowed")),
        ])
        .with_columns([
            (
                (
                    pl.col("prior_td_rate_allowed") * prior_games_equivalent +
                    pl.col("current_td_rate_allowed") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("td_rate_allowed"),

            (
                (
                    pl.col("prior_fg_rate_allowed") * prior_games_equivalent +
                    pl.col("current_fg_rate_allowed") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("fg_rate_allowed"),

            (
                (
                    pl.col("prior_turnover_rate_forced") * prior_games_equivalent +
                    pl.col("current_turnover_rate_forced") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("turnover_rate_forced"),

            (
                (
                    pl.col("prior_empty_rate_allowed") * prior_games_equivalent +
                    pl.col("current_empty_rate_allowed") * pl.col("current_games")
                )
                /
                (prior_games_equivalent + pl.col("current_games"))
            ).alias("empty_rate_allowed"),
        ])
    )

    return offense, defense

print("build_v1_pregame_ratings() ready.")


In [ ]:
# ============================================================
# CELL 9 — MATCHUP DRIVE PROBABILITIES
# ============================================================

def get_v1_matchup_probs(
    offense_team,
    defense_team,
    offense_ratings,
    defense_ratings
):
    off = (
        offense_ratings
        .filter(pl.col("team") == offense_team)
        .row(0, named=True)
    )

    defense = (
        defense_ratings
        .filter(pl.col("team") == defense_team)
        .row(0, named=True)
    )

    probs = {
        "TD": (off["td_rate"] + defense["td_rate_allowed"]) / 2,
        "FG": (off["fg_rate"] + defense["fg_rate_allowed"]) / 2,
        "TURNOVER": (off["turnover_rate"] + defense["turnover_rate_forced"]) / 2,
        "EMPTY": (off["empty_rate"] + defense["empty_rate_allowed"]) / 2,
    }

    total = sum(probs.values())

    return {
        outcome: prob / total
        for outcome, prob in probs.items()
    }

print("get_v1_matchup_probs() ready.")


In [ ]:
# ============================================================
# CELL 10 — LEAKAGE-SAFE LEAGUE ENVIRONMENT + HOME FIELD
# ============================================================

def build_v1_league_environment(
    target_season,
    target_week
):
    history = v1_drives.filter(
        (pl.col("season") < target_season)
        |
        (
            (pl.col("season") == target_season)
            &
            (pl.col("week") < target_week)
        )
    )

    if history.height == 0:
        raise ValueError("No historical data available before target.")

    game_drive_counts = (
        history
        .group_by("game_id")
        .agg(pl.len().alias("total_drives"))
    )

    local_drive_values = (
        game_drive_counts["total_drives"]
        .to_numpy()
        .astype(int)
    )

    venue_data = (
        history
        .with_columns(
            pl.when(pl.col("offense") == pl.col("home_team"))
              .then(pl.lit("HOME"))
              .otherwise(pl.lit("AWAY"))
              .alias("venue")
        )
        .with_columns([
            (pl.col("v1_outcome") == "TD").cast(pl.Float64).alias("_td"),
            (pl.col("v1_outcome") == "FG").cast(pl.Float64).alias("_fg"),
            (pl.col("v1_outcome") == "TURNOVER").cast(pl.Float64).alias("_turnover"),
            (pl.col("v1_outcome") == "EMPTY").cast(pl.Float64).alias("_empty"),
        ])
    )

    venue_summary = (
        venue_data
        .group_by("venue")
        .agg([
            pl.col("_td").mean().alias("td_rate"),
            pl.col("_fg").mean().alias("fg_rate"),
            pl.col("_turnover").mean().alias("turnover_rate"),
            pl.col("_empty").mean().alias("empty_rate"),
        ])
    )

    league_rates = {
        "TD": float(venue_data["_td"].mean()),
        "FG": float(venue_data["_fg"].mean()),
        "TURNOVER": float(venue_data["_turnover"].mean()),
        "EMPTY": float(venue_data["_empty"].mean()),
    }

    home = (
        venue_summary
        .filter(pl.col("venue") == "HOME")
        .row(0, named=True)
    )

    away = (
        venue_summary
        .filter(pl.col("venue") == "AWAY")
        .row(0, named=True)
    )

    home_multipliers = {
        "TD": home["td_rate"] / league_rates["TD"],
        "FG": home["fg_rate"] / league_rates["FG"],
        "TURNOVER": home["turnover_rate"] / league_rates["TURNOVER"],
        "EMPTY": home["empty_rate"] / league_rates["EMPTY"],
    }

    away_multipliers = {
        "TD": away["td_rate"] / league_rates["TD"],
        "FG": away["fg_rate"] / league_rates["FG"],
        "TURNOVER": away["turnover_rate"] / league_rates["TURNOVER"],
        "EMPTY": away["empty_rate"] / league_rates["EMPTY"],
    }

    return {
        "drive_values": local_drive_values,
        "home_multipliers": home_multipliers,
        "away_multipliers": away_multipliers,
        "games_used": len(local_drive_values),
    }


def apply_venue_adjustment(
    matchup_probs,
    venue_multipliers
):
    adjusted = {
        outcome: matchup_probs[outcome] * venue_multipliers[outcome]
        for outcome in matchup_probs
    }

    total = sum(adjusted.values())

    return {
        outcome: prob / total
        for outcome, prob in adjusted.items()
    }

print("League environment + venue functions ready.")


In [ ]:
# ============================================================
# CELL 11 — ODDS + SIMULATION + CALIBRATION HELPERS
# ============================================================

def prob_to_american(prob):
    if prob <= 0 or prob >= 1:
        return np.nan
    if prob >= 0.5:
        return round(-100 * prob / (1 - prob))
    return round(100 * (1 - prob) / prob)


def probability_to_logit(p):
    p = np.clip(
        np.asarray(p, dtype=float),
        0.001,
        0.999
    )
    return np.log(p / (1 - p))


def calculate_ece(
    df,
    prob_col,
    outcome_col="actual_home_win",
    bin_width=0.05
):
    bins = np.arange(0, 1 + bin_width, bin_width)

    temp = df.copy()
    temp["_bucket"] = pd.cut(
        temp[prob_col],
        bins=bins,
        include_lowest=True
    )

    calibration = (
        temp
        .groupby("_bucket", observed=True)
        .agg(
            games=(outcome_col, "size"),
            mean_prob=(prob_col, "mean"),
            actual_rate=(outcome_col, "mean")
        )
        .reset_index()
    )

    calibration["abs_error"] = (
        calibration["mean_prob"] -
        calibration["actual_rate"]
    ).abs()

    ece = (
        (
            calibration["games"] *
            calibration["abs_error"]
        ).sum()
        /
        calibration["games"].sum()
    )

    return float(ece), calibration


def simulate_v1_win_prob_fast(
    away_probs,
    home_probs,
    local_drive_values,
    n_simulations=10_000,
    seed=42
):
    rng = np.random.default_rng(seed)

    total_drives = rng.choice(
        local_drive_values,
        size=n_simulations,
        replace=True
    ).astype(int)

    base = total_drives // 2
    away_drives = base.copy()
    home_drives = base.copy()

    odd = total_drives % 2 == 1
    extra_home = odd & (rng.random(n_simulations) < 0.5)
    extra_away = odd & ~extra_home

    home_drives += extra_home.astype(int)
    away_drives += extra_away.astype(int)

    away_td = rng.binomial(away_drives, away_probs["TD"])
    away_remaining = away_drives - away_td
    away_fg_conditional = away_probs["FG"] / (1.0 - away_probs["TD"])
    away_fg = rng.binomial(away_remaining, away_fg_conditional)
    away_scores = 7 * away_td + 3 * away_fg

    home_td = rng.binomial(home_drives, home_probs["TD"])
    home_remaining = home_drives - home_td
    home_fg_conditional = home_probs["FG"] / (1.0 - home_probs["TD"])
    home_fg = rng.binomial(home_remaining, home_fg_conditional)
    home_scores = 7 * home_td + 3 * home_fg

    home_win = home_scores > away_scores
    away_win = away_scores > home_scores
    tie = home_scores == away_scores

    home_reg_prob = float(home_win.mean())
    away_reg_prob = float(away_win.mean())
    tie_prob = float(tie.mean())

    return {
        "home_ml_prob": home_reg_prob + 0.5 * tie_prob,
        "away_ml_prob": away_reg_prob + 0.5 * tie_prob,
        "reg_tie_prob": tie_prob,
        "mean_home_score": float(home_scores.mean()),
        "mean_away_score": float(away_scores.mean()),
    }

print("Odds, simulation, and calibration helpers ready.")


# V1.0 Historical Backtest

V1.0 is the raw baseline simulator.  
This section creates the historical probability set that V2.0 later calibrates.


In [ ]:
# ============================================================
# CELL 12 — V1.0 HISTORICAL BACKTEST
# 2019-2025, leakage-safe week by week
# ============================================================

BACKTEST_SEASONS = list(range(2019, 2026))
N_SIMS_BACKTEST = 10_000

backtest_rows = []
start_time = time.time()

for season in BACKTEST_SEASONS:

    season_games = (
        games
        .filter(pl.col("season") == season)
        .sort(["week", "game_id"])
    )

    weeks = sorted(
        season_games["week"]
        .unique()
        .to_list()
    )

    print(f"Backtesting V1.0 {season}...")

    for week in weeks:

        offense_ratings, defense_ratings = build_v1_pregame_ratings(
            target_season=season,
            target_week=week
        )

        env = build_v1_league_environment(
            target_season=season,
            target_week=week
        )

        local_drive_values = env["drive_values"]
        home_mult = env["home_multipliers"]
        away_mult = env["away_multipliers"]

        available_off = set(offense_ratings["team"].to_list())
        available_def = set(defense_ratings["team"].to_list())

        week_games = season_games.filter(
            pl.col("week") == week
        )

        for game_num, game in enumerate(
            week_games.iter_rows(named=True)
        ):

            home_team = game["home_team"]
            away_team = game["away_team"]

            if (
                home_team not in available_off
                or away_team not in available_off
                or home_team not in available_def
                or away_team not in available_def
            ):
                continue

            away_neutral = get_v1_matchup_probs(
                offense_team=away_team,
                defense_team=home_team,
                offense_ratings=offense_ratings,
                defense_ratings=defense_ratings
            )

            home_neutral = get_v1_matchup_probs(
                offense_team=home_team,
                defense_team=away_team,
                offense_ratings=offense_ratings,
                defense_ratings=defense_ratings
            )

            away_probs_bt = apply_venue_adjustment(
                away_neutral,
                away_mult
            )

            home_probs_bt = apply_venue_adjustment(
                home_neutral,
                home_mult
            )

            seed = season * 10000 + week * 100 + game_num

            sim = simulate_v1_win_prob_fast(
                away_probs=away_probs_bt,
                home_probs=home_probs_bt,
                local_drive_values=local_drive_values,
                n_simulations=N_SIMS_BACKTEST,
                seed=seed
            )

            if game["home_points"] > game["away_points"]:
                actual_home_win = 1
            elif game["home_points"] < game["away_points"]:
                actual_home_win = 0
            else:
                actual_home_win = None

            backtest_rows.append({
                "game_id": game["game_id"],
                "season": season,
                "week": week,
                "away_team": away_team,
                "home_team": home_team,
                "actual_away_points": game["away_points"],
                "actual_home_points": game["home_points"],
                "model_away_score": sim["mean_away_score"],
                "model_home_score": sim["mean_home_score"],
                "home_win_prob": sim["home_ml_prob"],
                "away_win_prob": sim["away_ml_prob"],
                "model_reg_tie_prob": sim["reg_tie_prob"],
                "actual_home_win": actual_home_win,
            })

backtest_v1 = pd.DataFrame(backtest_rows)
elapsed = time.time() - start_time

print()
print("V1.0 backtest complete.")
print("Games simulated:", len(backtest_v1))
print("Runtime:", round(elapsed, 2), "seconds")


In [ ]:
# ============================================================
# CELL 13 — V1.0 REPORT CARD
# ============================================================

decisive = (
    backtest_v1
    .dropna(subset=["actual_home_win"])
    .copy()
)

decisive["actual_home_win"] = decisive["actual_home_win"].astype(int)
decisive["home_win_prob_clipped"] = decisive["home_win_prob"].clip(0.001, 0.999)

favorite_accuracy = (
    ((decisive["home_win_prob"] > 0.5).astype(int) ==
     decisive["actual_home_win"])
    .mean()
)

brier = brier_score_loss(
    decisive["actual_home_win"],
    decisive["home_win_prob"]
)

ll = log_loss(
    decisive["actual_home_win"],
    decisive["home_win_prob_clipped"]
)

decisive["actual_margin"] = (
    decisive["actual_home_points"] -
    decisive["actual_away_points"]
)

decisive["model_margin"] = (
    decisive["model_home_score"] -
    decisive["model_away_score"]
)

margin_mae = np.mean(
    np.abs(
        decisive["model_margin"] -
        decisive["actual_margin"]
    )
)

print("NFL MONEYLINE V1.0 — HISTORICAL REPORT")
print("========================================")
print("Backtest seasons: 2019-2025")
print("Total games:", len(backtest_v1))
print("Decisive games:", len(decisive))
print("Winner accuracy:", round(favorite_accuracy, 4))
print("Brier score:", round(brier, 5))
print("Log loss:", round(ll, 5))
print("Projected-margin MAE:", round(margin_mae, 3))


# NFL Moneyline V2.0 — ACTIVE CHAMPION

**Only new concept:** leakage-safe probability calibration.

The football simulator is unchanged from V1.0.  
V2.0 learns how raw simulator probabilities historically translate into real win frequencies.


In [ ]:
# ============================================================
# CELL 14 — V2.0 ROLLING FORWARD CALIBRATION
# ============================================================

v2_rows = []
v2_calibration_parameters = []

for target_season in range(2021, 2026):

    train = decisive[
        decisive["season"] < target_season
    ].copy()

    test = decisive[
        decisive["season"] == target_season
    ].copy()

    X_train = probability_to_logit(
        train["home_win_prob"].values
    ).reshape(-1, 1)

    y_train = (
        train["actual_home_win"]
        .astype(int)
        .values
    )

    X_test = probability_to_logit(
        test["home_win_prob"].values
    ).reshape(-1, 1)

    calibrator = LogisticRegression(
        C=1_000_000,
        solver="lbfgs",
        max_iter=2000
    )

    calibrator.fit(X_train, y_train)

    test["v2_home_win_prob"] = (
        calibrator.predict_proba(X_test)[:, 1]
    )

    test["v2_away_win_prob"] = (
        1 - test["v2_home_win_prob"]
    )

    v2_rows.append(test)

    v2_calibration_parameters.append({
        "target_season": target_season,
        "training_games": len(train),
        "intercept": float(calibrator.intercept_[0]),
        "slope": float(calibrator.coef_[0][0]),
    })

backtest_v2 = pd.concat(
    v2_rows,
    ignore_index=True
)

v2_params = pd.DataFrame(
    v2_calibration_parameters
)

display(v2_params)


In [ ]:
# ============================================================
# CELL 15 — V1.0 VS V2.0
# Same 2021-2025 games
# ============================================================

comparison = backtest_v2.copy()
y = comparison["actual_home_win"].astype(int)

v1_probs_compare = comparison["home_win_prob"]
v2_probs_compare = comparison["v2_home_win_prob"]

v1_accuracy_compare = (
    ((v1_probs_compare >= 0.5).astype(int) == y)
    .mean()
)

v2_accuracy = (
    ((v2_probs_compare >= 0.5).astype(int) == y)
    .mean()
)

v1_brier_compare = brier_score_loss(y, v1_probs_compare)
v2_brier = brier_score_loss(y, v2_probs_compare)

v1_logloss_compare = log_loss(
    y,
    np.clip(v1_probs_compare, 0.001, 0.999)
)

v2_logloss = log_loss(
    y,
    np.clip(v2_probs_compare, 0.001, 0.999)
)

v1_ece_compare, _ = calculate_ece(
    comparison,
    "home_win_prob"
)

v2_ece, v2_calibration_table = calculate_ece(
    comparison,
    "v2_home_win_prob"
)

comparison_table = pd.DataFrame({
    "Metric": [
        "Games",
        "Winner Accuracy",
        "Brier Score",
        "Log Loss",
        "ECE",
    ],
    "V1.0 Raw": [
        len(comparison),
        v1_accuracy_compare,
        v1_brier_compare,
        v1_logloss_compare,
        v1_ece_compare,
    ],
    "V2.0 Calibrated": [
        len(comparison),
        v2_accuracy,
        v2_brier,
        v2_logloss,
        v2_ece,
    ],
})

display(comparison_table)

print()
print("V2.0 calibration table:")
display(v2_calibration_table)


In [ ]:
# ============================================================
# CELL 16 — FINAL V2.0 CALIBRATOR
# Train on all 2019-2025 decisive games
# ============================================================

final_train = decisive.copy()

X_final = probability_to_logit(
    final_train["home_win_prob"].values
).reshape(-1, 1)

y_final = (
    final_train["actual_home_win"]
    .astype(int)
    .values
)

v2_final_calibrator = LogisticRegression(
    C=1_000_000,
    solver="lbfgs",
    max_iter=2000
)

v2_final_calibrator.fit(
    X_final,
    y_final
)

final_intercept = float(
    v2_final_calibrator.intercept_[0]
)

final_slope = float(
    v2_final_calibrator.coef_[0][0]
)

print("NFL MONEYLINE V2.0 — FINAL CALIBRATOR")
print("======================================")
print("Training games:", len(final_train))
print("Intercept:", round(final_intercept, 6))
print("Slope:", round(final_slope, 6))


In [ ]:
# ============================================================
# CELL 17 — REUSABLE V2.0 MATCHUP ANALYZER
# ============================================================

def analyze_v20_matchup(
    away_team,
    home_team,
    target_season,
    target_week,
    n_simulations=100_000,
    seed=42
):
    offense_ratings, defense_ratings = build_v1_pregame_ratings(
        target_season=target_season,
        target_week=target_week
    )

    env = build_v1_league_environment(
        target_season=target_season,
        target_week=target_week
    )

    away_neutral = get_v1_matchup_probs(
        offense_team=away_team,
        defense_team=home_team,
        offense_ratings=offense_ratings,
        defense_ratings=defense_ratings
    )

    home_neutral = get_v1_matchup_probs(
        offense_team=home_team,
        defense_team=away_team,
        offense_ratings=offense_ratings,
        defense_ratings=defense_ratings
    )

    away_probs = apply_venue_adjustment(
        away_neutral,
        env["away_multipliers"]
    )

    home_probs = apply_venue_adjustment(
        home_neutral,
        env["home_multipliers"]
    )

    raw = simulate_v1_win_prob_fast(
        away_probs=away_probs,
        home_probs=home_probs,
        local_drive_values=env["drive_values"],
        n_simulations=n_simulations,
        seed=seed
    )

    home_raw_logit = probability_to_logit(
        [raw["home_ml_prob"]]
    ).reshape(-1, 1)

    home_calibrated = float(
        v2_final_calibrator
        .predict_proba(home_raw_logit)[0, 1]
    )

    away_calibrated = 1 - home_calibrated

    return {
        "away_team": away_team,
        "home_team": home_team,
        "mean_away_score": raw["mean_away_score"],
        "mean_home_score": raw["mean_home_score"],
        "raw_away_prob": raw["away_ml_prob"],
        "raw_home_prob": raw["home_ml_prob"],
        "away_prob": away_calibrated,
        "home_prob": home_calibrated,
        "away_fair_ml": prob_to_american(away_calibrated),
        "home_fair_ml": prob_to_american(home_calibrated),
        "reg_tie_prob": raw["reg_tie_prob"],
    }

print("analyze_v20_matchup() ready.")


In [ ]:
# ============================================================
# CELL 18 — CURRENT EXAMPLE: DEN @ KC
# Change these four values for another matchup
# ============================================================

TARGET_SEASON = 2026
TARGET_WEEK = 1
AWAY_TEAM = "DEN"
HOME_TEAM = "KC"

current_v20 = analyze_v20_matchup(
    away_team=AWAY_TEAM,
    home_team=HOME_TEAM,
    target_season=TARGET_SEASON,
    target_week=TARGET_WEEK,
    n_simulations=100_000,
    seed=42
)

print("NFL MONEYLINE V2.0")
print("===================")
print(
    f"Projected mean score: "
    f"{AWAY_TEAM} {current_v20['mean_away_score']:.2f} - "
    f"{HOME_TEAM} {current_v20['mean_home_score']:.2f}"
)
print()
print(
    f"{HOME_TEAM} calibrated ML probability:",
    round(current_v20["home_prob"], 4)
)
print(
    f"{HOME_TEAM} fair ML:",
    current_v20["home_fair_ml"]
)
print()
print(
    f"{AWAY_TEAM} calibrated ML probability:",
    round(current_v20["away_prob"], 4)
)
print(
    f"{AWAY_TEAM} fair ML:",
    current_v20["away_fair_ml"]
)


## Frozen V2.0 benchmark

Original research benchmark:

- V1.0 backtest: **2019–2025, 1,871 games**
- Decisive games: **1,865**
- V1.0 winner accuracy: **~64.34%**
- V1.0 Brier: **~0.22830**
- V1.0 log loss: **~0.64849**
- V1.0 margin MAE: **~10.379**

Forward-style V2.0 comparison window, 2021–2025:
- Games: **1,355**
- V2.0 Brier: **~0.22786**
- V2.0 log loss: **~0.64816**
- V2.0 ECE: **~4.59%**

Final V2.0 calibrator from the original run:
- Training games: **1,865**
- Intercept: **~-0.065053**
- Slope: **~1.52703**

If a clean rerun differs materially rather than by tiny Monte Carlo noise, investigate before adding new versions.


# NFL Moneyline V2.4 — Development Branch

**Base model:** V2.0 only.

Rejected experiments are intentionally absent:
- V2.1 pace — not included
- V2.2 field position — not included
- V2.3 turnover detail — not included

Anything added below this point belongs to V2.4 development.


In [ ]:
# ============================================================
# V2.4 STARTING CHECKPOINT
# ============================================================

print("NFL Moneyline V2.4 development branch")
print("Base: V2.0")
print("Rejected feature state loaded: NO")
print("Ready to design V2.4.")
